## 웹서비스 흐름
1. 클라이언트가 `http://example.com` 같은 URL로 요청을 보낸다.
2. 웹서버가 HTML 문서를 응답한다.
3. 웹브라우저는 HTML을 파싱해서 화면에 렌더링한다.

### 정적 웹페이지 스크래핑 전 확인
1. 대상 사이트의 이용약관을 확인한다.
2. `robots.txt`를 확인하여 크롤러 접근 정책을 확인한다.
3. `robots.txt`는 크롤러에게 “이 경로는 수집하지 말아 달라”고 알려주는 표준 정책 파일이다.
4. 단, `robots.txt`가 허용한다고 해서 저작권, 개인정보, 이용약관 문제가 모두 해결되는 것은 아니다.
5. 짧은 시간에 반복 요청을 보내면 서버에 부담을 줄 수 있으므로 요청 간격과 수집 범위를 조절한다.

### 정적 웹페이지란?
- 서버가 응답한 HTML 안에 필요한 데이터가 이미 포함된 페이지이다.
- `requests`로 HTML을 가져온 뒤 `BeautifulSoup`으로 원하는 태그를 찾을 수 있다.
- JavaScript 실행 후에 데이터가 생기는 페이지는 정적 방식만으로 수집이 어려울 수 있다.

In [2]:
# requests, beautifulsoup4 모듈 설치
!pip install requests beautifulsoup4

## 웹 페이지 요청 및 파싱

In [11]:
import requests
from bs4 import BeautifulSoup

url = 'https://www.naver.com'
# 일부 사이트는 브라우저가 아닌 요청을 제한할 수 있으므로 User-Agent를 함께 전달한다.
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/125.0 Safari/537.36"
}

try:
    response = requests.get(url, headers=headers, timeout=10)   # 요청을 보낸다.get 메서드 방식으로(url을, 헤더와, 타임아웃은 10초)

    # 400, 500번대 응답 상태 코드 확인 시 에러 발생
    response.raise_for_status()
except requests.exceptions.Timeout:
    print("시간 초과")
except requests.exceptions.RequestException as e:
    print("요청 실패:", e)

else:
    # 정상 응답 시
    html = response.text
    # print(html)

    # BeautifulSoup 객체를 이용한 파싱 작업
    soup = BeautifulSoup(html, 'html.parser')
    print(soup.title)

<title>NAVER</title>


## find | find_all

In [33]:
# 내가 가진 html 파싱
from bs4 import BeautifulSoup

with open('sample.html', 'r', encoding='utf-8') as f:
    html = f.read() # sample.html 내용 읽어오기 -> html 모양의 문자열

    # html 문자열 파싱
    soup = BeautifulSoup(html, 'html.parser')
    # print(soup.title)

    # find_all: 모든 일치하는 태그 조회
    li_tags = soup.find_all('li')
    # print(li_tags)
    for li in li_tags:
        print(type(li), li.text)

    print("-" * 50)

    # find: 일치하는 첫 번째 태그만 조회
    first_li = soup.find('li')
    print("first_li:", first_li.text)

    print("-" * 50)

    # find('태그명', {속성명 : 속성값}): 첫 번째 요소 찾기
    h1_tag = soup.find('h1', {"id" : "page-title"})
                    # h1이라는 곳에서 id가 page-title인 걸 가져오겠다.
    print("h1_tag:", h1_tag.text)

    # find_all('태그명', {속성명 : 속성값}): 모든 요소 찾기
    section_content_tags = soup.find_all('section', {"class" : "section-content"})
    for section in section_content_tags:
        print("section.text:", section.text)    # 내용
        print("section.attrs:", section.attrs)  # 속성

<class 'bs4.element.Tag'> Section 1
<class 'bs4.element.Tag'> Section 2
<class 'bs4.element.Tag'> Section 3
<class 'bs4.element.Tag'> List item 1
<class 'bs4.element.Tag'> List item 2
<class 'bs4.element.Tag'> List item 3
<class 'bs4.element.Tag'> First item
<class 'bs4.element.Tag'> Second item
<class 'bs4.element.Tag'> Third item
--------------------------------------------------
first_li: Section 1
--------------------------------------------------
h1_tag: Welcome to the Sample HTML Page
section.text: 
Section 1: Introduction

            This is a sample paragraph.
            Bold text
            and
            italic text
            are commonly used to emphasize content. You can also include
            links
            to other pages.
        
Here is an image:


section.attrs: {'id': 'section1', 'class': ['section-content']}
section.text: 
Section 2: List Examples
Unordered List

List item 1
List item 2
List item 3

Ordered List

First item
Second item
Third item


section

## select | select_one
- CSS 선택자('태그명', '#', '.', '>' 등 기호)를 이용하여 태그 조회

In [47]:
li_tags = soup.select('li')
# print(li_tags)

first_li = soup.select_one('li')
# print(first_li)

# id(#)를 이용해서 찾기
h1_tag = soup.select_one('#page-title')
print(h1_tag)

# class(.)를 이용해서 찾기
section_content_tags = soup.select('.section-content')
print(len(section_content_tags))

# 자식 요소 찾기 (부모 > 자식)    부모에서 자식 쪽으로 들어감
h2_tags = soup.select('.section-content > h2')  # 자식 쪽으로 들어가 h2를 찾겠다.
print(h2_tags)

# 후손(하위 모든 요소) 찾기  (부모 후손)  ('띄어쓰기'가 선택자 기호)
strong_tag = soup.select('.section-content strong')
print(strong_tag)

print("-" * 50)
strong_tag = soup.select('.section-content strong')
# 이전 형제 찾기: find_previous_sibling()


# 다음 형제 찾기: find_next_sibling()
em_tag = soup.select_one('.section-content em')
print(em_tag)
print(em_tag.find_previous_sibling())
print(em_tag.find_next_sibling())

<h1 id="page-title">Welcome to the Sample HTML Page</h1>
3
[<h2>Section 1: Introduction</h2>, <h2>Section 2: List Examples</h2>, <h2>Section 3: Table Example</h2>]
[<strong class="highlight">Bold text</strong>]
--------------------------------------------------
<em>italic text</em>
<strong class="highlight">Bold text</strong>
<a href="https://www.example.com" target="_blank">links</a>


## 네이버 뉴스 검색 결과에서 제목만 스크래핑하기

In [53]:
import requests
from bs4 import BeautifulSoup as bs

query = '인공지능'  # 검색어
url = f'https://search.naver.com/search.naver?ssc=tab.news.all&where=news&sm=tab_jum&query={query}'

try:
    response = requests.get(url, timeout=10)
    response.raise_for_status()

except requests.exceptions.Timeout:
    print("시간 초과")
except requests.exceptions.RequestException as e:
    print("오류 발생", e)

else:
    soup = bs(response.text, 'html.parser') # 받은 데이터를 파싱함.

    # 파싱된 html 코드에서 '뉴스 제목 찾기'
    title_tags = soup.select('.sds-comps-text.sds-comps-text-ellipsis.sds-comps-text-ellipsis-1.sds-comps-text-type-headline1')

    titles = [ tag.get_text(strip = True) for tag in title_tags]  # strip = true: 좌우공백제거
    for title in titles:
        print(title)

KAIST,인공지능최대 난제 '발열' 잡을 냉각기술 확보
SK바이오팜, 바이오 USA서인공지능(AI) 신약개발 전략 공개
“사번 001, AI 과장입니다”… SKT ‘인공지능동료’ 파격 실험
인공지능·디지털 교육 거점 3곳으로 확대
폴리텍IV대학 대전캠퍼스, 전국인공지능드론 경진서 대상·장려상
신동빈 회장, "AX는 기업 생존위한 절대 과제". 롯데그룹,인공지능전환...
인공지능시대 뒤처질라…"中 대학 학위과정 30% 넘게 조정"
인공지능투자 열풍에 반도체 주도...5월 수출물가 11개월째 상승
울산시,인공지능선박 기술 주도권 확보 시동
비투엔, 글로벌 기업 ‘오두’와 맞손…인공지능전사적자원관리 시업...


In [56]:
# 뉴스 검색 결과에서 기사 링크만 스크래핑
a_tags = soup.select(".sds-comps-vertical-layout.sds-comps-full-layout > a:first-of-type")

links = [tag.get('href') for tag in a_tags]

print('수집된 링크 개수:', len(links))
print(links)

수집된 링크 개수: 10
['https://www.newsis.com/view/NISX20260616_0003670583', 'https://www.hankyung.com/article/202606168441i', 'https://www.munhwa.com/article/11595889?ref=naver', 'https://news.kbs.co.kr/news/pc/view/view.do?ncd=8587188&ref=A', 'https://www.news1.kr/local/daejeon-chungnam/6198491', 'https://www.autodaily.co.kr/news/articleView.html?idxno=544924', 'https://www.yna.co.kr/view/AKR20260615114900009?input=1195m', 'https://www.m-economynews.com/news/article.html?no=67934', 'https://www.ksilbo.co.kr/news/articleView.html?idxno=1059571', 'https://www.etoday.co.kr/news/view/2593895']


In [57]:
# 기사 설명
description_tags = soup.select('span.sds-comps-text.sds-comps-text-ellipsis.sds-comps-text-ellipsis-3.sds-comps-text-type-body1')
descriptions = [span.get_text(' ', strip=True) for span in description_tags]

print(f'수집된 설명 개수: {len(descriptions)}')
print(descriptions)

수집된 설명 개수: 10
['인공지능 (A)I 데이터센터의 냉각전력을 90%나 줄 일 수 있는 발열관리기술이 나왔다. 한국과학기술원(카이스트·KAIST)는 기계공학과 김성진 교수팀과 AX학과 이익진 교수팀이 공동연구를 통해 반도체 칩 내부에 머리카락보다 가는 물길을 새겨 넣는 초고효율 액체 냉각기술을 개발해 AI 반도체의 최대...', 'SK바이오팜은 이번 행사에서 다수의 글로벌 제약사 및 투자자들과의 파트너링 미팅을 통해 신규 협력 기회를 모색하고, 부스를 통해 연구개발과 회사 운영 전반에 걸친 인공지능 (AI) 활용 방향을 소개할 예정이다. BIO USA는 美 바이오협회(BIO)가 주관하는 세계 최대 규모의 바이오 산업 행사다. 올해...', 'SK텔레콤이 인공지능 (AI) 에이전트에 사번을 부여하고 소속과 직무·권한까지 할당하는 파격적인 실험에 나서 눈길을 끌고 있다. SK텔레콤은 구성원이 AI 전환(AX)에 몰입할 수 있는 환경을 구축하기 위해 이 같은 내용을 담은 ‘AX 혁신 2.0’을 시행한다고 16일 밝혔다. 현장의 업무 효율성을 개선하는 데...', '울산시가 시민 누구나 인공지능 과 디지털 기술을 배우고 활용할 수 있도록 관련 교육을 확대합니다. 이에따라 무인 안내기와 드론, 디지털 건강관리 기기 등을 직접 체험할 수 있는 공간을 운영하고 기관과 단체의 신청을 받아 강사를 파견하는 교육 서비스도 제공합니다. 울산시는 시민 접근성을 높이기 위해...', "한국폴리텍IV대학(학장 양형규) 대전캠퍼스 스마트소프트웨어과는 전국 규모의 인공지능 드론 경진대회에서 대상과 장려상을 수상했다고 16일 밝혔다. 스마트소프트웨어과는 최근 부천대학교 한길아트홀에서 열린 '2026 IES 산업전자 기술대전' 부대행사인 '제9회 전국대학교 인공지능 드론 경진대회'에...", '롯데그룹이 전사적인 인공지능 전환(AX·AI Transformation)에 속도를 내고 있다. 신동빈 롯데그룹 회장이 직접 AI 교육 프로그램에 참여하며 그룹 차원의 AI 혁신을 강력하게 주문한

## 스크래핑한 기사를 저장할 클래스

In [58]:
from datetime import datetime

class NaverNews:
    def __init__(self, id: int, title: str, originallink: str, link: str, description: str, pub_date: str,
                 created_at: datetime = None):
        # id는 DB의 auto_increment 기본키를 담기 위한 값이다.
        # 스크래핑 직후에는 아직 DB에 저장되지 않았으므로 None을 사용할 수 있다.
        self.__id = id
        self.__title = title
        self.__originallink = originallink
        self.__link = link
        self.__description = description

        # 네이버 API 응답 key는 pubDate이지만, Python 객체 내부에서는 pub_date 이름을 사용한다.
        self.__pub_date = pub_date
        self.__created_at = created_at

    @classmethod
    def from_api_item(cls, item: dict):
        """네이버 API 응답 item의 pubDate key를 pub_date 속성으로 변환한다."""

        return cls(
            id=None,
            title=item.get('title', ''),
            originallink=item.get('originallink', ''),
            link=item.get('link', ''),
            description=item.get('description', ''),
            pub_date=item.get('pubDate', ''),
        )

    @property
    def id(self):
        return self.__id

    @property
    def title(self):
        return self.__title

    # __title 속성에 대한 setter 메소드
    # 아래 주석을 해제하면 naver_news.title = '새 제목'처럼 값을 변경할 수 있다.
    # @title.setter
    # def title(self, value):
    #     self.__title = value

    @property
    def originallink(self):
        return self.__originallink

    @property
    def link(self):
        return self.__link

    @property
    def description(self):
        return self.__description

    @property
    def pub_date(self):
        """Python 코드에서 사용할 발행일 속성."""

        return self.__pub_date

    @property
    def created_at(self):
        return self.__created_at

    def __repr__(self):
        return f'NaverNews({self.__id}, {self.__title}, {self.__originallink}, {self.__link}, {self.__description}, {self.__pub_date}, {self.__created_at})'

### 스크래핑한 titles, links, descriptions를 묶어서 네이버 뉴스 객체 리스트로 변환

In [61]:
print(len(titles), len(links), len(descriptions))

# zip: 같은 인덱스끼리 묶어줌 (적은 숫자 기준)
news_lines = zip(titles, links, descriptions)

naver_news_list: list[NaverNews] = []

for title, link, descriptions in news_lines:
    naver_news_list.append(
        NaverNews(
            id = None,
            title = title,
            originallink = link,
            link = link,
            description = descriptions,
            pub_date = None
        )
    )

for news in naver_news_list:
    print(news)

10 10 180
NaverNews(None, KAIST,인공지능최대 난제 '발열' 잡을 냉각기술 확보, https://www.newsis.com/view/NISX20260616_0003670583, https://www.newsis.com/view/NISX20260616_0003670583, 이, None, None)
NaverNews(None, SK바이오팜, 바이오 USA서인공지능(AI) 신약개발 전략 공개, https://www.hankyung.com/article/202606168441i, https://www.hankyung.com/article/202606168441i, 투, None, None)
NaverNews(None, “사번 001, AI 과장입니다”… SKT ‘인공지능동료’ 파격 실험, https://www.munhwa.com/article/11595889?ref=naver, https://www.munhwa.com/article/11595889?ref=naver, 데, None, None)
NaverNews(None, 인공지능·디지털 교육 거점 3곳으로 확대, https://news.kbs.co.kr/news/pc/view/view.do?ncd=8587188&ref=A, https://news.kbs.co.kr/news/pc/view/view.do?ncd=8587188&ref=A, 이, None, None)
NaverNews(None, 폴리텍IV대학 대전캠퍼스, 전국인공지능드론 경진서 대상·장려상, https://www.news1.kr/local/daejeon-chungnam/6198491, https://www.news1.kr/local/daejeon-chungnam/6198491, =, None, None)
NaverNews(None, 신동빈 회장, "AX는 기업 생존위한 절대 과제". 롯데그룹,인공지능전환..., https://www.autodaily.co.kr/news/articleView.html?idxno=544924, http

## 이미지 스크래핑

In [63]:
import os

parent_dir = './naver_news_images'
os.makedirs(parent_dir, exist_ok=True)  # exist_ok=true : 없으면 생성을 의미

image_tags = soup.select(".sds-comps-base-layout.sds-comps-full-layout img")

for index, image_tag in enumerate(image_tags):
    src = image_tag.get('src')  # 이미지 주소 얻어오기

    try:
        image_response = requests.get(src, timeout=10)
        image_response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print("오류 발생: ", e)

    # 저장할 경로
    file_path = os.path.join(
        parent_dir,
        f"{index}.jpg"
    )
    with open(file_path, 'wb') as f:    # 파일이 없을 때 하면 0byte짜리 파일이 만들어짐
        f.write(image_response.content)

print("이미지 저장 완료")

이미지 저장 완료
